# 10 — Evidence-Grounded Prompting and RAG Interfaces

## Scenario
Northstar must answer questions about refund policies. We want to avoid hallucination, so we require the model to ground its answers in factual evidence.

**The Problem:** LLMs are eager to please and will often invent plausible-sounding policies if they don't know the answer.

## Step 1: Ungrounded Generation (Baseline)

Watch what happens when we ask a niche question without any grounding.

## Step 2: Manual Grounding (Classic RAG)

We "retrieve" a document (mocked here) and strictly instruct the model to use it.

## Step 3: Managed Grounding with Google Search (State of the Art)

Instead of manually pasting text into prompts, modern APIs natively support grounding endpoints. Here we demonstrate using Google Search natively via the Gemini API to ground an answer and receive structured citations.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab10 import *

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: Ungrounded Generation (Baseline)

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i10/ungrounded/custom-mug")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)

## Step 2: Manual Grounding (Classic RAG)

In [ ]:
evidence = retrieve(client, "custom mug Northstar logo")
print("RETRIEVED:", evidence)
request = next(r for r in build_requests() if r.case_id == "i10/final/custom-mug")
print("PROMPT:", request.messages[0].text)
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
assert check_citations(response.text, evidence).unknown_ids == set()

## Step 3: Managed Grounding with a Retrieval Tool

In [ ]:
request = next(r for r in build_requests() if r.case_id == "i10/tool/custom-mug")
print("SYSTEM:\n", request.system)
print("USER:\n", request.messages[0].text)
tool_response = client.generate(request)
print("RECORDED TOOL CALL:", tool_response.tool_calls)
assert tool_response.tool_calls[0].name == "search_policies"
answer = client.generate(next(r for r in build_requests() if r.case_id == "i10/final/no-support"))
print("ABSTENTION:", answer.text)
assert is_abstention(answer.text)

## Takeaway

This replay-backed experiment makes the application control and measured trade-off explicit.

## References

See the course README for the references and further reading.